In [ ]:
from langgraph.graph import StateGraph, START,END
from typing import TypedDict, Literal
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import operator 
load_dotenv()

generator_llm = ChatGroq(model="llama-3.1-8b-instant")
evaluator_llm = ChatGroq(model="llama-3.1-8b-instant")
optimizer_llm = ChatGroq(model="llama-3.1-8b-instant")

In [ ]:
class TweetState(TypedDict):
    title: str
    tweet: str
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str
    iteration: int
    max_iteration: int

    tweet_history: Annotated[list[str], operator.add]
    feedback_history: Annonated[list[str], operator.add]

In [ ]:
def TweetEvaluationSchema(BaseModel):
    evaluation: Literal['approved', 'needs_improvement'] = Field(description="Answer in these two fields only")
    feedback: str = Field(description="Contains feedback of the evaluation")

In [ ]:
structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluationSchema)

In [ ]:
def generate_tweet(state: TweetState):
    prompt = f"generate a tweet according to the title {state['title']}"
    twe = generator_llm.invoke(prompt)
    return {'tweet': twe, 'tweet_history': list[twe]}

In [ ]:
def evaluate_tweet(state: TweetState):
    prompt: f"evaluate the tweet {state['tweet']} with title {state['title']}"
    response = structured_evaluator_llm.invoke(prompt)
    return {"evaluation": response.evaluation, "feedback": response.feedback, 'feedback_history': list[response.feedback]}

In [ ]:
def optimise_tweet(state: TweetState):
    prompt = f"Optimise the tweet {state['tweet']} according to the feedback {state['feedback']}"
    tweet = generator_llm.invoke(prompt)
    iteration= state['iteration']+1
    return {'tweet': tweet, 'iteration': iteration}


In [ ]:
def route_evaluation(state: TweetState):

    if state['evaluation'] == "approved" or state['iteration']>5:
        return 'approved'
    else:
        return 'needs_improvement'

In [ ]:
graph = StateGraph(TweetState)

graph.add_node("generate_tweet", generate_tweet)
graph.add_node("evaluate_tweet", evaluate_tweet)
graph.add_node("optimise_tweet", optimise_tweet)

In [ ]:
#  here we will loop we are only manupilate the edges
graph.add_edge(START, "generate_tweet")
graph.add_edge("generate_tweet", "evaluate_tweet")
graph.add_conditional_edges("evaluate_tweet", route_evaluation, {"approved": END, "needs_improvement": optimise_tweet})
graph.add_edge("optimise_tweet", "evaluate_tweet")

workflow = graph.compile()
initial_state= {
    "topic": "Indian Railways",
    "iteration":1,
    "max_iteration": 5
}

final_state = workflow.invoke(initial_state)

SyntaxError: invalid syntax (1140210113.py, line 1)